好的！下面是《动手学深度学习》第11.9节 **Adadelta算法** 的完整学习笔记，已整理为 **Markdown 格式**，你可以直接复制粘贴进 Word 或 Jupyter Notebook 使用。

---

# 📘 11.9 Adadelta算法 - 学习笔记

## 📌 一、Adadelta 是什么？

**Adadelta** 是对 AdaGrad 的进一步改进，旨在解决 AdaGrad 学习率衰减太快、RMSProp仍需手动指定初始学习率的问题。

> 它是 **一个完全自适应学习率的优化算法**，**不需要设置初始学习率 \$\eta\$**。

---

## ⚙️ 二、核心思想与公式

Adadelta 的设计思路是：

* 和 RMSProp 一样，使用 **梯度平方的滑动平均**（代替累加的平方和）
* 同时，对 **参数更新量本身** 也维护一个滑动平均（用于归一化）

---

### 🔢 更新公式

定义两个滑动平均变量：

* \$\boldsymbol{s}\_t\$：梯度平方的滑动平均
* \$\boldsymbol{\Delta x}\_t^2\$：更新步长平方的滑动平均

计算步骤如下：

1. **更新梯度平方滑动平均：**

$$
\boldsymbol{s}_t = \gamma \boldsymbol{s}_{t-1} + (1 - \gamma) \boldsymbol{g}_t^2
$$

2. **计算更新量：**

$$
\Delta \boldsymbol{x}_t = - \frac{\sqrt{\boldsymbol{\Delta x}_{t-1}^2 + \epsilon}}{\sqrt{\boldsymbol{s}_t + \epsilon}} \odot \boldsymbol{g}_t
$$

3. **更新参数：**

$$
\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} + \Delta \boldsymbol{x}_t
$$

4. **更新步长平方的滑动平均：**

$$
\boldsymbol{\Delta x}_t^2 = \gamma \boldsymbol{\Delta x}_{t-1}^2 + (1 - \gamma) \Delta \boldsymbol{x}_t^2
$$

---

### 🧠 直观解释

* 和 RMSProp 类似，Adadelta 使用滑动平均对梯度大小进行“平滑”，避免学习率过快下降。
* 与 RMSProp 不同，**Adadelta 用历史步长来自适应调整更新幅度**，不依赖手动设置学习率。
* 它使得每一维参数的单位一致（步长/梯度都归一化），更自然。

---

## 🧪 三、代码实现（from scratch）

```python
def adadelta(params, states, hyperparams):
    rho, eps = hyperparams['rho'], 1e-5
    for (p, (s, delta)) in zip(params, states):
        with torch.no_grad():
            s[:] = rho * s + (1 - rho) * torch.square(p.grad)
            g = (torch.sqrt(delta + eps) / torch.sqrt(s + eps)) * p.grad
            p[:] -= g
            delta[:] = rho * delta + (1 - rho) * torch.square(g)
        p.grad.zero_()
```

### 初始化状态：

```python
def init_adadelta_states(feature_dim):
    s = torch.zeros((feature_dim, 1))
    delta = torch.zeros((feature_dim, 1))
    return (s, delta)
```

---

## 📦 四、PyTorch简洁实现

```python
trainer = torch.optim.Adadelta(model.parameters(), rho=0.9)
```

无需设置学习率，`rho=0.9` 是默认的滑动平均系数。

---

## 📊 五、与其它算法对比

| 优化器          | 是否使用学习率 \$\eta\$ | 学习率变化机制      | 特点与适用场景 |
| ------------ | ---------------- | ------------ | ------- |
| SGD          | ✅                | 固定或手动调节      | 简单粗暴    |
| AdaGrad      | ✅                | 梯度平方累计导致快速下降 | 稀疏特征数据  |
| RMSProp      | ✅                | 梯度平方滑动平均     | 常用通用型   |
| **Adadelta** | ❌                | 自动基于历史步长归一化  | 更加稳定与鲁棒 |

---

## 📝 六、总结要点

* Adadelta 是对 RMSProp 的改进版本，**完全自适应学习率**。
* 使用 **梯度平方滑动平均 + 步长滑动平均** 实现归一化。
* 不依赖手动设置学习率，**在多数情况下表现稳定、无需调参**。
* 为后续优化器（如 Adam）的提出打下了基础。

---

下一节（11.10）是最重要的 **Adam优化算法**，它结合了 RMSProp 与动量法的优点。如果你准备好了，我可以继续为你整理 **Adam 的 Markdown 学习笔记**。是否继续？


你问的 `rho` 是 **Adadelta 算法中的超参数**，它控制的是：

> 👉 **滑动平均中的“保留历史信息的比例”**

---

## 📌 `rho` 是什么？

在 Adadelta 中，有两个滑动平均：

1. 梯度平方的滑动平均：\$\boldsymbol{s}\_t\$
2. 更新步长的滑动平均：\$\boldsymbol{\Delta x}\_t^2\$

它们的更新方式如下：

$$
\boldsymbol{s}_t = \rho \cdot \boldsymbol{s}_{t-1} + (1 - \rho) \cdot \boldsymbol{g}_t^2
$$

$$
\boldsymbol{\Delta x}_t^2 = \rho \cdot \boldsymbol{\Delta x}_{t-1}^2 + (1 - \rho) \cdot \Delta \boldsymbol{x}_t^2
$$

这里的 \$\rho\$ 就是你问的参数。

---

## 🧠 通俗理解：

* `rho` 类似于\*\*“惯性”**或**“记忆因子”\*\*
* 如果 `rho` 越大（如 0.99）：

  * 越依赖过去的梯度
  * 趋势更加平稳，变化缓慢
* 如果 `rho` 越小（如 0.5）：

  * 更快响应新梯度，但可能噪声大、不稳定

---

## 🧪 默认值

在 `PyTorch` 中，`Adadelta` 的 `rho` 默认值是：

```python
torch.optim.Adadelta(..., rho=0.9)
```

也就是说，**当前状态由 90% 旧状态 + 10% 新梯度贡献**。

---

## ✅ 总结一句话：

> `rho` 决定了 Adadelta 对“历史信息”的保留程度，越大则越平滑，越小则对当前梯度反应更敏感。

---

如果你还对 `rho` 的数值怎么选、是否需要调参等问题感兴趣，我也可以接着展开讲讲\~
